In [1]:
import pandas as pd
import os
import glob
import numpy as np
import json

In [2]:
CSV_PATH = "/data/rbg/users/ashyamal/RNA-FM-benchmarking/hp_results/tuning_history.csv"
LOG_DIR = "/data/rbg/users/ashyamal/RNA-FM-benchmarking/hp_results"

RNA_KEYS = ["BulkFormer", "BulkRNABert", "COMPASS", "scGPT", "STATE", "CancerFoundation", "RNA_seq_1250", "RNA_seq_17490", "ssGSEA_combined", "max_combined", "PCA_50"]
ARCHS = ["ResidualMLP", "Linear"]
LOSSES = ["Cox", "CIndex"]
SPLITS = [0, 1, 2, 3, 4]
N_TRIALS = 50

# Load CSV
df = pd.read_csv(CSV_PATH)

# Count completed trials per (key, arch, loss, split)
completed = df.groupby(["rna_embedding_key", "arch", "loss_name", "split_idx"]).size().reset_index(name="n_trials")

# Build full expected set
expected = pd.MultiIndex.from_product(
    [RNA_KEYS, ARCHS, LOSSES, SPLITS],
    names=["rna_embedding_key", "arch", "loss_name", "split_idx"]
).to_frame(index=False)

# Merge to find missing/incomplete
merged = expected.merge(completed, on=["rna_embedding_key", "arch", "loss_name", "split_idx"], how="left")
merged["n_trials"] = merged["n_trials"].fillna(0).astype(int)
merged["complete"] = merged["n_trials"] >= N_TRIALS
merged["over"] = merged["n_trials"] > N_TRIALS

print(f"Complete: {merged['complete'].sum()} / {len(merged)}")
print(f"Over-complete: {merged['over'].sum()} / {len(merged)}\n")
print(f"Missing/incomplete:\n")

incomplete = merged[~merged["complete"]].sort_values(["rna_embedding_key", "arch", "loss_name", "split_idx"])
print(incomplete.to_string(index=False))

Complete: 220 / 220
Over-complete: 37 / 220

Missing/incomplete:

Empty DataFrame
Columns: [rna_embedding_key, arch, loss_name, split_idx, n_trials, complete, over]
Index: []


In [3]:
over_complete = merged[merged["over"]].sort_values(["rna_embedding_key", "arch", "loss_name", "split_idx"])
print(over_complete.to_string(index=False))

rna_embedding_key        arch loss_name  split_idx  n_trials  complete  over
      BulkRNABert      Linear    CIndex          0        61      True  True
      BulkRNABert      Linear    CIndex          1        62      True  True
      BulkRNABert      Linear    CIndex          3        62      True  True
      BulkRNABert      Linear    CIndex          4        61      True  True
      BulkRNABert      Linear       Cox          0        60      True  True
      BulkRNABert      Linear       Cox          1        61      True  True
      BulkRNABert      Linear       Cox          2        60      True  True
      BulkRNABert      Linear       Cox          3        61      True  True
      BulkRNABert      Linear       Cox          4        61      True  True
          COMPASS      Linear    CIndex          0        66      True  True
          COMPASS      Linear    CIndex          1        67      True  True
          COMPASS      Linear    CIndex          2        67      True  True

In [4]:
# For each over-complete group, keep only the first 50 trials
def trim_to_n_trials(group, n=50):
    return group.head(n)

df_trimmed = df.groupby(
    ["rna_embedding_key", "arch", "loss_name", "split_idx"],
    group_keys=False
).apply(trim_to_n_trials)

/tmp/ipykernel_4069949/779304962.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(trim_to_n_trials)


In [5]:
print(f"Trimmed DataFrame has {len(df_trimmed)} from original {len(df)}")

Trimmed DataFrame has 11000 from original 11719


In [6]:
# For each (key, arch, loss, split), get the best trial (min val_loss)
best_per_split = df.loc[df.groupby(["rna_embedding_key", "arch", "loss_name", "split_idx"])["val_loss"].idxmin()]

# Aggregate best HPs across splits
log_hps = ["lr", "weight_decay", "penalty_weight"]
linear_hps = ["dropout_rate"]

def aggregate_group(group):
    result = {}
    for hp in log_hps:
        vals = group[hp].replace("", np.nan).dropna().astype(float)
        result[hp] = float(np.exp(np.mean(np.log(vals)))) if len(vals) > 0 else np.nan
    for hp in linear_hps:
        vals = group[hp].replace("", np.nan).dropna().astype(float)
        result[hp] = float(np.mean(vals)) if len(vals) > 0 else np.nan
    return pd.Series(result)

aggregated_hps = best_per_split.groupby(["rna_embedding_key", "arch", "loss_name"]).apply(aggregate_group).reset_index()

print(aggregated_hps.to_string(index=False))

rna_embedding_key        arch loss_name       lr  weight_decay  penalty_weight  dropout_rate
       BulkFormer      Linear    CIndex 0.000899  1.877586e-06        0.000133           NaN
       BulkFormer      Linear       Cox 0.000609  3.067406e-06        0.000227           NaN
       BulkFormer ResidualMLP    CIndex 0.000218  1.333806e-06        0.000127      0.160145
       BulkFormer ResidualMLP       Cox 0.000134  3.019898e-06        0.000137      0.189237
      BulkRNABert      Linear    CIndex 0.000956  1.326303e-06        0.000108           NaN
      BulkRNABert      Linear       Cox 0.000980  2.037617e-06        0.000127           NaN
      BulkRNABert ResidualMLP    CIndex 0.000445  3.031032e-06        0.000127      0.225973
      BulkRNABert ResidualMLP       Cox 0.000234  7.195769e-06        0.000162      0.237776
          COMPASS      Linear    CIndex 0.000891  2.042548e-06        0.000107           NaN
          COMPASS      Linear       Cox 0.000968  1.089237e-06        

/tmp/ipykernel_4069949/76767742.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  aggregated_hps = best_per_split.groupby(["rna_embedding_key", "arch", "loss_name"]).apply(aggregate_group).reset_index()


In [7]:
all_hps = {}

for _, row in aggregated_hps.iterrows():
    key = f"{row['rna_embedding_key']}_{row['arch']}_{row['loss_name']}"
    all_hps[key] = {
        "lr": row["lr"],
        "weight_decay": row["weight_decay"],
        "penalty_weight": row["penalty_weight"],
    }
    if row["arch"] != "Linear":
        all_hps[key]["dropout_rate"] = row["dropout_rate"]

out_path = "/data/rbg/users/ashyamal/RNA-FM-benchmarking/hp_results/all_aggregated_hps.json"
with open(out_path, "w") as f:
    json.dump(all_hps, f, indent=2)

print(f"Saved to {out_path}")

Saved to /data/rbg/users/ashyamal/RNA-FM-benchmarking/hp_results/all_aggregated_hps.json
